# El evaluador dentro del bucle: el mejor de k semillas y el asaltante sin modelo

Cuaderno de lectura de la medición `evaluador-bucle/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda el artículo [quien-vigila-al-evaluador](https://manpla.net/posts/quien-vigila-al-evaluador/). Carga los ficheros de al lado —o los descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja dos figuras con matplotlib a secas. Solo lee; no vuelve a simular: para eso está `generar.py`, que tarda medio minuto.

*Reading notebook for this measurement: loads the files next to it (or from the repository), prints the provenance note and draws two plain matplotlib figures. It only reads; `generar.py` re-runs the simulations.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/evaluador-bucle/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")

## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El mejor de k semillas

In [ ]:
s = leer("semillas.csv")
s[(s.p == 0.7) & (s.n.isin([100, 500, 5000]))].pivot(index="k", columns="n", values="inflacion_simulada_puntos")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for n, g in s[s.p == 0.7].groupby("n"):
    ax.plot(g.k, g.inflacion_simulada_puntos, marker="o", label=f"n = {n}")
ax.set_xscale("log"); ax.set_xlabel("k evaluaciones entre las que se elige la mejor"); ax.set_ylabel("puntos de más, p = 0,7"); ax.legend(); plt.tight_layout()

## El asaltante sin modelo

In [ ]:
a = leer("asalto-serie.csv")
fig, ax = plt.subplots(figsize=(7, 4))
for c in ("ingenuo", "escalera", "rotado", "anclado", "real"):
    ax.plot(a.consulta, a[c], label=c)
ax.set_xlabel("consultas al evaluador"); ax.set_ylabel("% publicado (n = 1000)"); ax.legend(); plt.tight_layout()
leer("asalto-por-tamano.csv")